[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tranthithuvan555-wq/TEP/blob/main/Self_Attention_Transformer_GPT2.ipynb)

# Xây dựng cơ chế tự chú ý và kiến trúc máy biến áp (GPT-2) từ đầu

Notebook này hướng dẫn từng bước xây dựng:
1. Cơ chế **Self-Attention** (Tự chú ý)
2. **Multi-Head Attention**
3. Kiến trúc **Transformer** đầy đủ
4. Mô hình **GPT-2** đơn giản từ đầu

Tất cả đều được triển khai bằng **PyTorch** với chú thích tiếng Việt.

## 0. Cài đặt thư viện cần thiết

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install torch numpy matplotlib

In [ ]:
# Import các thư viện
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# Đặt seed để tái lập kết quả
torch.manual_seed(42)
np.random.seed(42)

print('Phiên bản PyTorch:', torch.__version__)
print('Thiết bị:', 'GPU' if torch.cuda.is_available() else 'CPU')

---
## 1. Giới thiệu về Self-Attention (Cơ chế tự chú ý)

### Self-Attention là gì?

**Self-Attention** (hay còn gọi là Intra-Attention) là cơ chế cho phép mỗi từ trong câu "chú ý" đến tất cả các từ khác (kể cả chính nó) để học được biểu diễn ngữ cảnh phong phú hơn.

**Ví dụ:** Trong câu *"Con mèo ngồi trên tấm thảm vì nó mệt"*, từ "nó" cần liên kết với "con mèo". Self-attention giúp mô hình học được mối liên kết này.

### Query, Key, Value là gì?

Mỗi từ được biến đổi thành 3 vector:
- **Query (Q):** "Tôi đang tìm kiếm thông tin gì?"
- **Key (K):** "Tôi có thể cung cấp thông tin gì?"
- **Value (V):** "Nội dung thực sự của tôi là gì?"

### Công thức Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Trong đó:
- $Q$: Ma trận Query, kích thước $(n, d_k)$
- $K$: Ma trận Key, kích thước $(n, d_k)$
- $V$: Ma trận Value, kích thước $(n, d_v)$
- $d_k$: Số chiều của Key (dùng để chia tỷ lệ, tránh gradient vanishing)

### Sơ đồ minh họa Self-Attention

```
  Input X
     │
  ┌──┴──┐
  │     │     │
 W_q   W_k   W_v
  │     │     │
  Q     K     V
  │     │     │
  └──┬──┘     │
   QK^T / √d_k│
     │         │
  SoftMax      │
     │         │
  Attention ───┘
  Weights
     │
   Output
```

---
## 2. Triển khai Self-Attention từ đầu bằng PyTorch

### 2.1 Single-Head Attention

In [ ]:
class SingleHeadAttention(nn.Module):
    """
    Triển khai Single-Head Self-Attention.
    
    Tham số:
        d_model (int): Số chiều của embedding đầu vào
        d_k (int): Số chiều của Query và Key
        d_v (int): Số chiều của Value
    """
    def __init__(self, d_model, d_k, d_v):
        super(SingleHeadAttention, self).__init__()
        
        # Các ma trận chiếu tuyến tính W_q, W_k, W_v
        self.W_q = nn.Linear(d_model, d_k, bias=False)  # Chiếu sang Query
        self.W_k = nn.Linear(d_model, d_k, bias=False)  # Chiếu sang Key
        self.W_v = nn.Linear(d_model, d_v, bias=False)  # Chiếu sang Value
        
        self.d_k = d_k  # Lưu lại để tính tỷ lệ
    
    def forward(self, x, mask=None):
        """
        x: tensor đầu vào, kích thước (batch_size, seq_len, d_model)
        mask: mặt nạ tùy chọn để che một số vị trí
        """
        # Tính Query, Key, Value
        Q = self.W_q(x)  # (batch_size, seq_len, d_k)
        K = self.W_k(x)  # (batch_size, seq_len, d_k)
        V = self.W_v(x)  # (batch_size, seq_len, d_v)
        
        # Tính điểm attention: QK^T / sqrt(d_k)
        # K.transpose(-2, -1) chuyển vị chiều cuối cùng
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores: (batch_size, seq_len, seq_len)
        
        # Áp dụng mask nếu có (dùng cho causal attention trong GPT)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax để có attention weights (tổng bằng 1)
        attention_weights = F.softmax(scores, dim=-1)
        # attention_weights: (batch_size, seq_len, seq_len)
        
        # Nhân attention weights với Value để lấy đầu ra
        output = torch.matmul(attention_weights, V)
        # output: (batch_size, seq_len, d_v)
        
        return output, attention_weights


# Ví dụ minh họa với câu tiếng Việt
print('=== Ví dụ Single-Head Attention ===')
print('Câu ví dụ: "Con mèo ngồi trên thảm"')

# Giả sử mỗi từ được biểu diễn bằng vector 8 chiều
batch_size = 1
seq_len = 5   # 5 từ: Con, mèo, ngồi, trên, thảm
d_model = 8   # Số chiều embedding
d_k = 4       # Số chiều Key/Query
d_v = 4       # Số chiều Value

# Tạo embedding giả cho câu tiếng Việt
x = torch.randn(batch_size, seq_len, d_model)

# Khởi tạo và chạy mô hình
attention = SingleHeadAttention(d_model, d_k, d_v)
output, weights = attention(x)

print(f'Kích thước đầu vào: {x.shape}  → (batch, seq_len, d_model)')
print(f'Kích thước đầu ra: {output.shape} → (batch, seq_len, d_v)')
print(f'Kích thước attention weights: {weights.shape} → (batch, seq_len, seq_len)')

In [ ]:
# Trực quan hóa attention weights
words = ['Con', 'mèo', 'ngồi', 'trên', 'thảm']

plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='Attention Weight')
plt.xticks(range(seq_len), words, fontsize=11)
plt.yticks(range(seq_len), words, fontsize=11)
plt.xlabel('Key (từ được chú ý tới)', fontsize=12)
plt.ylabel('Query (từ đang chú ý)', fontsize=12)
plt.title('Attention Weights\n"Con mèo ngồi trên thảm"', fontsize=13)
plt.tight_layout()
plt.show()
print('Mỗi hàng thể hiện mức độ chú ý của một từ tới các từ khác trong câu.')

### 2.2 Multi-Head Attention

**Multi-Head Attention** chạy nhiều cơ chế attention song song, cho phép mô hình học nhiều loại mối quan hệ khác nhau giữa các từ cùng một lúc.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

Trong đó: $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Triển khai Multi-Head Self-Attention.
    
    Tham số:
        d_model (int): Số chiều embedding
        num_heads (int): Số lượng attention head
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, 'd_model phải chia hết cho num_heads'
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Số chiều mỗi head
        
        # Ma trận chiếu cho Q, K, V và đầu ra
        self.W_q = nn.Linear(d_model, d_model)  # Chiếu Query
        self.W_k = nn.Linear(d_model, d_model)  # Chiếu Key
        self.W_v = nn.Linear(d_model, d_model)  # Chiếu Value
        self.W_o = nn.Linear(d_model, d_model)  # Chiếu đầu ra
    
    def split_heads(self, x):
        """Chia tensor thành nhiều heads.
        (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, _ = x.shape
        # Reshape và hoán vị chiều
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Chiếu và chia thành nhiều heads
        Q = self.split_heads(self.W_q(x))  # (batch, heads, seq_len, d_k)
        K = self.split_heads(self.W_k(x))  # (batch, heads, seq_len, d_k)
        V = self.split_heads(self.W_v(x))  # (batch, heads, seq_len, d_k)
        
        # Tính attention score cho tất cả heads cùng lúc
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        
        # Tính đầu ra của từng head
        context = torch.matmul(attention_weights, V)  # (batch, heads, seq_len, d_k)
        
        # Ghép tất cả heads lại
        context = context.transpose(1, 2).contiguous()     # (batch, seq_len, heads, d_k)
        context = context.view(batch_size, seq_len, self.d_model)  # (batch, seq_len, d_model)
        
        # Chiếu đầu ra
        output = self.W_o(context)  # (batch, seq_len, d_model)
        
        return output, attention_weights


# Kiểm tra Multi-Head Attention
print('=== Ví dụ Multi-Head Attention ===')
d_model = 64
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
output, weights = mha(x)

print(f'Số lượng heads: {num_heads}')
print(f'Số chiều mỗi head: {d_model // num_heads}')
print(f'Kích thước đầu vào:  {x.shape}')
print(f'Kích thước đầu ra:   {output.shape}')
print(f'Kích thước weights:  {weights.shape}  → (batch, heads, seq, seq)')

---
## 3. Kiến trúc Transformer

### 3.1 Positional Encoding (Mã hóa vị trí)

Transformer không có khái niệm thứ tự tự nhiên như RNN, nên cần thêm thông tin vị trí vào embedding:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Mã hóa vị trí dùng hàm sin/cos.
    Thêm thông tin về vị trí của mỗi token vào embedding.
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Tạo ma trận PE kích thước (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        
        # Vị trí từ 0 đến max_len-1
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Hệ số chia tỷ lệ
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        
        # Điền giá trị sin/cos xen kẽ
        pe[:, 0::2] = torch.sin(position * div_term)  # Các chiều chẵn: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # Các chiều lẻ: cos
        
        # Thêm chiều batch và đăng ký là buffer (không phải tham số học)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # Cộng positional encoding vào embedding
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Trực quan hóa Positional Encoding
d_model = 64
pe_layer = PositionalEncoding(d_model, dropout=0)
pe_values = pe_layer.pe[0].numpy()  # (max_len, d_model)

plt.figure(figsize=(10, 4))
plt.imshow(pe_values[:50, :].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(label='Giá trị PE')
plt.xlabel('Vị trí trong chuỗi', fontsize=12)
plt.ylabel('Chiều embedding', fontsize=12)
plt.title('Positional Encoding (50 vị trí đầu tiên)', fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Feed-Forward Network và Layer Normalization

In [ ]:
class FeedForwardNetwork(nn.Module):
    """
    Mạng nơ-ron truyền thẳng (Feed-Forward Network) trong Transformer.
    Gồm 2 lớp tuyến tính với hàm kích hoạt GELU ở giữa.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForwardNetwork, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)   # Mở rộng chiều
        self.linear2 = nn.Linear(d_ff, d_model)   # Thu hẹp lại
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x → Linear(d_model, d_ff) → GELU → Dropout → Linear(d_ff, d_model)
        x = self.linear1(x)
        x = F.gelu(x)          # Hàm kích hoạt GELU (GPT-2 dùng GELU)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


# Kiểm tra FFN
ffn = FeedForwardNetwork(d_model=64, d_ff=256)
x = torch.randn(2, 10, 64)
out = ffn(x)
print(f'FFN: đầu vào {x.shape} → đầu ra {out.shape}')
print('Lớp mở rộng giữa:', ffn.linear1.weight.shape, '→ tỷ lệ mở rộng 4x')

---
## 4. Xây dựng GPT-2 từ đầu

### Kiến trúc GPT-2

GPT-2 là **Decoder-only Transformer** với các đặc điểm:
- Chỉ sử dụng phần Decoder (không có Encoder)
- Dùng **Causal (Autoregressive) Attention**: mỗi token chỉ nhìn được các token đứng trước nó
- Mục tiêu: Dự đoán token tiếp theo dựa trên các token đã có

```
Input tokens
    │
Token Embedding + Positional Embedding
    │
┌───┴──────────────┐
│  Transformer Block × N  │
│  ┌──────────────┐  │
│  │ LayerNorm    │  │
│  │ Masked MHA   │  │
│  │ + Residual   │  │
│  │ LayerNorm    │  │
│  │ FFN          │  │
│  │ + Residual   │  │
│  └──────────────┘  │
└───────────────────┘
    │
LayerNorm
    │
Linear (Language Model Head)
    │
Logits (vocab_size)
```

In [ ]:
class GPT2Block(nn.Module):
    """
    Một Transformer Block trong GPT-2.
    Sử dụng Pre-LayerNorm (chuẩn hóa trước attention và FFN).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(GPT2Block, self).__init__()
        
        # Chuẩn hóa lớp (Pre-LN: chuẩn hóa trước mỗi lớp con)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Multi-Head Attention với causal mask
        self.attn = MultiHeadAttention(d_model, num_heads)
        
        # Feed-Forward Network
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        seq_len = x.size(1)
        
        # Tạo causal mask: token tại vị trí i chỉ nhìn được vị trí 0..i
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, seq)
        
        # Attention với residual connection và Pre-LN
        attn_out, _ = self.attn(self.ln1(x), mask=causal_mask)
        x = x + self.dropout(attn_out)  # Residual connection
        
        # FFN với residual connection và Pre-LN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)   # Residual connection
        
        return x


class SimpleGPT2(nn.Module):
    """
    Mô hình GPT-2 đơn giản.
    
    Tham số:
        vocab_size (int): Kích thước từ điển
        d_model (int): Số chiều embedding
        num_heads (int): Số lượng attention heads
        num_layers (int): Số lượng Transformer blocks
        d_ff (int): Số chiều lớp ẩn trong FFN
        max_seq_len (int): Độ dài chuỗi tối đa
    """
    def __init__(self, vocab_size, d_model=128, num_heads=4,
                 num_layers=4, d_ff=512, max_seq_len=256, dropout=0.1):
        super(SimpleGPT2, self).__init__()
        
        # Embedding token và vị trí
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # Ngăn xếp các Transformer Blocks
        self.blocks = nn.ModuleList([
            GPT2Block(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Chuẩn hóa cuối cùng
        self.ln_f = nn.LayerNorm(d_model)
        
        # Đầu dự đoán ngôn ngữ (Language Model Head)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Chia sẻ trọng số giữa embedding và lm_head (giống GPT-2 gốc)
        self.lm_head.weight = self.token_embedding.weight
        
        # Khởi tạo trọng số
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Khởi tạo trọng số theo chuẩn GPT-2."""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)
    
    def forward(self, idx):
        """
        idx: tensor chỉ số token, kích thước (batch_size, seq_len)
        """
        batch_size, seq_len = idx.shape
        
        # Tạo vị trí: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, device=idx.device).unsqueeze(0)
        
        # Token embedding + Positional embedding
        tok_emb = self.token_embedding(idx)          # (batch, seq_len, d_model)
        pos_emb = self.position_embedding(positions)  # (1, seq_len, d_model)
        x = self.dropout(tok_emb + pos_emb)
        
        # Đi qua các Transformer Blocks
        for block in self.blocks:
            x = block(x)
        
        # Chuẩn hóa cuối
        x = self.ln_f(x)
        
        # Dự đoán xác suất token tiếp theo
        logits = self.lm_head(x)  # (batch, seq_len, vocab_size)
        
        return logits


# Thống kê mô hình
vocab_size = 1000
model = SimpleGPT2(vocab_size=vocab_size, d_model=128, num_heads=4,
                   num_layers=4, d_ff=512, max_seq_len=256)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=== Thống kê mô hình GPT-2 đơn giản ===')
print(f'Kích thước từ điển:    {vocab_size}')
print(f'Số chiều embedding:    128')
print(f'Số attention heads:    4')
print(f'Số Transformer layers: 4')
print(f'Tổng số tham số:       {total_params:,}')
print(f'Tham số có thể học:    {trainable_params:,}')

### 4.1 Demo: Sinh văn bản (Text Generation)

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_ids, max_new_tokens=20, temperature=1.0, top_k=10):
    """
    Sinh văn bản tự động bằng mô hình GPT-2.
    
    Tham số:
        model: Mô hình GPT-2
        prompt_ids: Tensor chỉ số token đầu vào (1, seq_len)
        max_new_tokens: Số token tối đa cần sinh
        temperature: Điều chỉnh độ ngẫu nhiên (cao → ngẫu nhiên hơn)
        top_k: Lấy k token có xác suất cao nhất để chọn ngẫu nhiên
    """
    model.eval()
    generated = prompt_ids.clone()
    
    for step in range(max_new_tokens):
        # Giới hạn độ dài chuỗi đầu vào
        context = generated[:, -256:]
        
        # Dự đoán logits
        logits = model(context)          # (1, seq_len, vocab_size)
        logits = logits[:, -1, :]        # Lấy logits của vị trí cuối cùng
        
        # Áp dụng temperature
        logits = logits / temperature
        
        # Top-k sampling: chỉ giữ k token có điểm cao nhất
        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            min_value = values[:, -1].unsqueeze(-1)
            logits = logits.masked_fill(logits < min_value, float('-inf'))
        
        # Softmax để lấy xác suất
        probs = F.softmax(logits, dim=-1)
        
        # Lấy mẫu token tiếp theo
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Nối token mới vào chuỗi
        generated = torch.cat([generated, next_token], dim=1)
    
    return generated


# Demo sinh văn bản (dùng từ điển giả)
print('=== Demo Sinh Văn Bản ===')
print('(Chú ý: Mô hình chưa được huấn luyện, nên output là ngẫu nhiên)')
print()

# Tạo prompt giả (ví dụ: 5 token đầu vào)
prompt = torch.randint(0, vocab_size, (1, 5))
print(f'Token đầu vào (prompt): {prompt[0].tolist()}')

# Sinh thêm 15 token
generated = generate_text(model, prompt, max_new_tokens=15, temperature=0.8, top_k=10)
new_tokens = generated[0, 5:].tolist()

print(f'Token được sinh ra:     {new_tokens}')
print(f'Tổng chuỗi token:       {generated[0].tolist()}')
print()
print('Để có kết quả có nghĩa, cần:')
print('  1. Xây dựng từ điển thực (vocab) từ tập văn bản tiếng Việt')
print('  2. Huấn luyện mô hình trên corpus tiếng Việt lớn')
print('  3. Điều chỉnh hyperparameters (d_model, num_layers, ...)')

---
## 5. Tổng kết

### Những gì chúng ta đã học:

| Thành phần | Chức năng |
|---|---|
| **Single-Head Attention** | Học mối quan hệ giữa các từ theo một cách |
| **Multi-Head Attention** | Học nhiều loại quan hệ song song |
| **Positional Encoding** | Thêm thông tin vị trí vào embedding |
| **Feed-Forward Network** | Xử lý từng vị trí độc lập, học biểu diễn phi tuyến |
| **Layer Normalization** | Ổn định quá trình huấn luyện |
| **Residual Connection** | Tránh vanishing gradient, giúp học sâu hơn |
| **GPT-2 Block** | Kết hợp tất cả thành phần trên |
| **Language Model Head** | Dự đoán token tiếp theo |

### Bước tiếp theo:
- Xem **Notebook 2** để học cách sử dụng các mô hình pre-trained như BERT và GPT-2
- Thử fine-tune mô hình trên tập dữ liệu tiếng Việt của bạn

### Tài liệu tham khảo:
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
- [Language Models are Unsupervised Multitask Learners (GPT-2)](https://openai.com/research/language-unsupervised)
- [The Illustrated Transformer (Jay Alammar)](http://jalammar.github.io/illustrated-transformer/)